# Model 2: Swin Transformer for Eye Disease Detection

This notebook provides the implementation, loading, and evaluation pipeline for **Swin Transformer** (Swin-Tiny) applied to fundus image screening across 6 pathology categories.

In [ ]:
import os
import torch
import torch.nn as nn
from torchvision import transforms
import timm

CLASSES = ['AMD', 'Cataract', 'Dementia', 'Diabetes', 'Glaucoma', 'Normal']
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
WEIGHT_PATH = 'weights/swin_scratch_best.pth'

print(f"Device: {DEVICE}")

## 1. Load Swin Transformer Model & Weight File

In [ ]:
def build_swin_transformer(num_classes=6):
    model = timm.create_model('swin_tiny_patch4_window7_224', pretrained=False, num_classes=num_classes)
    return model

model = build_swin_transformer(len(CLASSES))
if os.path.exists(WEIGHT_PATH):
    state_dict = torch.load(WEIGHT_PATH, map_location=DEVICE)
    model.load_state_dict(state_dict)
    print(f"Loaded weights from: {WEIGHT_PATH}")
else:
    print(f"Weights file {WEIGHT_PATH} not found.")

model = model.to(DEVICE)
model.eval()

## 2. Evaluation & Inference Benchmark

In [ ]:
dummy_input = torch.randn(1, 3, 224, 224).to(DEVICE)
with torch.no_grad():
    outputs = model(dummy_input)
    probs = torch.softmax(outputs, dim=1)
    pred_idx = torch.argmax(probs, dim=1).item()

print(f"Predicted pathology: {CLASSES[pred_idx]} (Confidence: {probs[0][pred_idx].item()*100:.2f}%)")